In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd drive/MyDrive/

/content/drive/MyDrive


In [ ]:
train = pd.read_csv('traffic_V3.csv')
test = pd.read_csv('test_traffic_V3.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (249944, 94)
테스트 데이터 크기: (50000, 93)


In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        task_type='CPU',
        verbose=100
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
0:	learn: 14.1949263	test: 14.2519939	best: 14.2519939 (0)	total: 151ms	remaining: 2m 30s
100:	learn: 9.3654096	test: 9.4536873	best: 9.4536873 (100)	total: 10.7s	remaining: 1m 35s
200:	learn: 9.2491769	test: 9.3769795	best: 9.3769795 (200)	total: 22.2s	remaining: 1m 28s
300:	learn: 9.1380344	test: 9.3066206	best: 9.3066206 (300)	total: 32s	remaining: 1m 14s
400:	learn: 9.0384422	test: 9.2473514	best: 9.2473514 (400)	total: 42s	remaining: 1m 2s
500:	learn: 8.9511819	test: 9.1975283	best: 9.1975283 (500)	total: 51.2s	remaining: 51s
600:	learn: 8.8695990	test: 9.1507656	best: 9.1507656 (600)	total: 1m 1s	remaining: 40.6s
700:	learn: 8.7933625	test: 9.1073939	best: 9.1073939 (700)	total: 1m 11s	remaining: 30.4s
800:	learn: 8.7257716	test: 9.0691717	best: 9.0691717 (800)	total: 1m 21s	remaining: 20.2s
900:	learn: 8.6676888	test: 9.0399244	best: 9.0399244 (900)	total: 1m 31s	remaining: 10.1s
999:	learn: 8.6136303	test: 9.0112775	best: 9.0112775 (999)	total: 1m 41s	remaining: 0u

In [ ]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 8.9683


In [ ]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V17.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
